# Shoealls — WearGait-PD 학습 (Google Colab)

**필요 사항:**
- Google Drive에 `WearGait-PD/HC/`, `WearGait-PD/PD/` 폴더가 업로드되어 있어야 합니다.
- GitHub Personal Access Token (repo 접근 권한)
- 런타임 → 런타임 유형 변경 → **T4 GPU** 선택


In [ ]:
# 1. GPU 확인
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# 2. Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Drive에서 데이터 경로 확인
import os

DRIVE_DATA = '/content/drive/MyDrive/WearGait-PD'
HC_DIR = os.path.join(DRIVE_DATA, 'HC')
PD_DIR = os.path.join(DRIVE_DATA, 'PD')

hc_files = [f for f in os.listdir(HC_DIR) if f.endswith('.csv')]
pd_files = [f for f in os.listdir(PD_DIR) if f.endswith('.csv')]
print(f'HC: {len(hc_files)}개, PD: {len(pd_files)}개')

In [ ]:
# 4. GitHub에서 레포 클론 (Private repo → 토큰 입력)
from google.colab import userdata
import subprocess

# Colab Secrets에 GITHUB_TOKEN을 등록하거나 아래 직접 입력
# 왼쪽 자물쇠 아이콘 → Secrets → GITHUB_TOKEN 추가
try:
    token = userdata.get('GITHUB_TOKEN')
except:
    token = input('GitHub Personal Access Token 입력: ').strip()

REPO = 'jg-shoealls/Shoealls'  # 실제 repo 경로로 변경
!git clone https://{token}@github.com/{REPO}.git /content/Shoealls 2>&1 | tail -3

import sys
sys.path.insert(0, '/content/Shoealls')
print('레포 클론 완료')

In [ ]:
# 5. 의존성 설치
!pip install -q pandas scipy scikit-learn pyyaml matplotlib
print('설치 완료')

In [ ]:
# 6. 데이터 로드
import numpy as np
from pathlib import Path

# Drive 데이터를 /content/data/raw 로 심볼릭 링크
os.makedirs('/content/data/raw', exist_ok=True)
if not os.path.exists('/content/data/raw/HC'):
    os.symlink(HC_DIR, '/content/data/raw/HC')
if not os.path.exists('/content/data/raw/PD'):
    os.symlink(PD_DIR, '/content/data/raw/PD')

os.chdir('/content/Shoealls')

from src.data.weargait_adapter import WearGaitPDAdapter

adapter = WearGaitPDAdapter('/content/data/raw')

# SelfPace 태스크 로드 (HurriedPace도 가능)
TASK = 'SelfPace'
dataset_dict = adapter.load_all(task=TASK)

labels = dataset_dict['labels']
print(f'\n총 샘플: {len(labels)}개')
print(f'HC(0): {(labels==0).sum()}개, PD(3): {(labels==3).sum()}개')

In [ ]:
# 7. 설정 로드 및 Dataset 생성
import yaml
import torch
from torch.utils.data import DataLoader, random_split

CONFIG_PATH = '/content/Shoealls/configs/default.yaml'
if os.path.exists(CONFIG_PATH):
    with open(CONFIG_PATH) as f:
        config = yaml.safe_load(f)
else:
    config = {
        'data': {
            'sequence_length': 128,
            'imu_channels': 6,
            'pressure_grid_size': [16, 8],
            'skeleton_joints': 17,
            'skeleton_dims': 3,
            'num_classes': 4,
            'class_names': ['Normal', 'Antalgic', 'Ataxic', 'Parkinsonian'],
        },
        'model': {
            'fusion': {'embed_dim': 128},
            'imu_encoder': {'dropout': 0.1},
            'pressure_encoder': {'dropout': 0.1},
            'skeleton_encoder': {'gcn_channels': [64, 128], 'temporal_kernel': 9, 'dropout': 0.1},
        },
        'training': {
            'batch_size': 16,
            'epochs': 30,
            'learning_rate': 1e-3,
            'weight_decay': 1e-4,
        }
    }

config['data']['num_classes'] = 4

dataset = adapter.to_dataset(sequence_length=config['data']['sequence_length'])
print(f'Dataset 생성: {len(dataset)}개 샘플')

total = len(dataset)
train_n = int(total * 0.7)
val_n   = int(total * 0.15)
test_n  = total - train_n - val_n
train_ds, val_ds, test_ds = random_split(
    dataset, [train_n, val_n, test_n],
    generator=torch.Generator().manual_seed(42)
)
print(f'학습: {len(train_ds)} / 검증: {len(val_ds)} / 테스트: {len(test_ds)}')

BS = config['training']['batch_size']
train_loader = DataLoader(train_ds, batch_size=BS, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BS, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BS, num_workers=2)

In [ ]:
# 8. 모델 초기화
import torch.nn as nn
from src.models.multimodal_gait_net import MultimodalGaitNet

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

model = MultimodalGaitNet(config).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f'모델 파라미터: {total_params:,}개')

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config['training']['learning_rate'],
    weight_decay=config['training']['weight_decay']
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=config['training']['epochs']
)

In [ ]:
# 9. 학습 루프
import time

EPOCHS = config['training']['epochs']
SAVE_DIR = '/content/drive/MyDrive/WearGait-PD/checkpoints'
os.makedirs(SAVE_DIR, exist_ok=True)

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc = 0.0

def run_epoch(loader, train=True):
    model.train(train)
    total_loss, correct, total = 0, 0, 0
    with torch.set_grad_enabled(train):
        for batch in loader:
            imu      = batch['imu'].to(device)
            pressure = batch['pressure'].to(device)
            skeleton = batch['skeleton'].to(device)
            labels   = batch['label'].to(device)

            out = model(imu, pressure, skeleton)
            logits = out if isinstance(out, torch.Tensor) else out['logits']
            loss = criterion(logits, labels)

            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            total_loss += loss.item() * len(labels)
            correct    += (logits.argmax(1) == labels).sum().item()
            total      += len(labels)

    return total_loss / total, correct / total

print(f'{'Epoch':>6} | {'Train Loss':>10} | {'Train Acc':>9} | {'Val Loss':>8} | {'Val Acc':>7}')
print('-' * 55)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    vl_loss, vl_acc = run_epoch(val_loader,   train=False)
    scheduler.step()

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(vl_acc)

    print(f'{epoch:>6} | {tr_loss:>10.4f} | {tr_acc:>9.4f} | {vl_loss:>8.4f} | {vl_acc:>7.4f}  ({time.time()-t0:.1f}s)')

    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'config': config,
            'val_accuracy': best_val_acc,
            'model_type': 'basic'
        }, os.path.join(SAVE_DIR, 'best_model.pt'))
        print(f'  ★ Best model saved (val_acc={best_val_acc:.4f})')

print(f'\n학습 완료! 최고 Val Acc: {best_val_acc:.4f}')

In [ ]:
# 10. 최종 테스트 평가
checkpoint = torch.load(os.path.join(SAVE_DIR, 'best_model.pt'))
model.load_state_dict(checkpoint['model_state_dict'])

test_loss, test_acc = run_epoch(test_loader, train=False)
print(f'테스트 정확도: {test_acc:.4f} ({test_acc*100:.1f}%)')
print(f'테스트 Loss:   {test_loss:.4f}')

In [ ]:
# 11. 학습 곡선 시각화
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['train_loss'], label='Train')
ax1.plot(history['val_loss'],   label='Val')
ax1.set_title('Loss')
ax1.set_xlabel('Epoch')
ax1.legend()
ax1.grid(True)

ax2.plot(history['train_acc'], label='Train')
ax2.plot(history['val_acc'],   label='Val')
ax2.set_title('Accuracy')
ax2.set_xlabel('Epoch')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'training_curve.png'), dpi=150)
plt.show()
print('학습 곡선이 Drive에 저장되었습니다.')

In [ ]:
# 12. 샘플별 예측 확인
CLASS_NAMES = ['Normal', 'Antalgic', 'Ataxic', 'Parkinsonian']
CLASS_NAMES_KR = ['정상 보행', '절뚝거림', '운동실조', '파킨슨']

model.eval()
sample_batch = next(iter(test_loader))
with torch.no_grad():
    imu      = sample_batch['imu'].to(device)
    pressure = sample_batch['pressure'].to(device)
    skeleton = sample_batch['skeleton'].to(device)
    out = model(imu, pressure, skeleton)
    logits = out if isinstance(out, torch.Tensor) else out['logits']
    probs  = torch.softmax(logits, dim=-1).cpu().numpy()
    preds  = probs.argmax(axis=1)
    gts    = sample_batch['label'].numpy()

print(f'{'샘플':>4} | {'실제':^10} | {'예측':^10} | {'신뢰도':>6} | 정오')
print('-' * 50)
for i in range(min(10, len(gts))):
    ok = '✓' if preds[i] == gts[i] else '✗'
    print(f'{i+1:>4} | {CLASS_NAMES_KR[gts[i]]:^10} | {CLASS_NAMES_KR[preds[i]]:^10} | {probs[i][preds[i]]:.1%} | {ok}')